In [15]:
import sys
sys.path.append("..")  # So we can import from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# Notebook display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

print("✅ Imports successful")

✅ Imports successful


In [ ]:
import importlib
import src.data_loader as dl
importlib.reload(dl)
from src.data_loader import load_season
from src.data_loader import load_season

# Load all three seasons
df_2023 = load_season(2023)
df_2024 = load_season(2024)
df_2025 = load_season(2025)

# Combine into one master dataframe
df = pd.concat([df_2023, df_2024, df_2025], ignore_index=True)

print(f"Total laps loaded: {len(df):,}")
print(f"Seasons: {df['Year'].unique()}")
print(f"Tracks: {df['EventName'].nunique()} unique events")
print(f"Drivers: {df['Driver'].nunique()} unique drivers")
print(f"\nColumns: {list(df.columns)}")

FileNotFoundError: No saved data for 2023. Run collect_season() first.

In [ ]:
# Shape and basic info
print(f"Dataset shape: {df.shape}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing %", ascending=False)

# Only show columns that actually have missing values
print(missing_df[missing_df["Missing Count"] > 0].to_string())

In [ ]:
# Convert LapTime from timedelta to seconds
df["LapTimeSeconds"] = df["LapTime"].dt.total_seconds()

# Same for sector times
df["Sector1Seconds"] = df["Sector1Time"].dt.total_seconds()
df["Sector2Seconds"] = df["Sector2Time"].dt.total_seconds()
df["Sector3Seconds"] = df["Sector3Time"].dt.total_seconds()

# Quick sanity check — F1 laps are typically 60-120 seconds
print("Lap time stats (seconds):")
print(df["LapTimeSeconds"].describe())

In [ ]:
total_before = len(df)

df_clean = df[
    # Remove laps with no recorded time
    df["LapTimeSeconds"].notna() &
    
    # Remove in-laps and out-laps (pit stop laps)
    (~df["PitInTime"].notna()) &
    (~df["PitOutTime"].notna()) &
    
    # Remove unrealistically slow laps (safety car, red flag, first lap)
    # Anything over 150 seconds is almost certainly not a real flying lap
    (df["LapTimeSeconds"] < 150) &
    
    # Remove unrealistically fast laps (data errors)
    (df["LapTimeSeconds"] > 50) &
    
    # Remove lap 1 (standing start, unpredictable)
    (df["LapNumber"] > 1) &
    
    # Only keep laps where tyre data is available
    df["Compound"].notna()
].copy()

total_after = len(df_clean)
removed = total_before - total_after

print(f"Laps before cleaning : {total_before:,}")
print(f"Laps after cleaning  : {total_after:,}")
print(f"Laps removed         : {removed:,} ({removed/total_before*100:.1f}%)")

In [ ]:
fig = px.histogram(
    df_clean,
    x="LapTimeSeconds",
    nbins=100,
    color="Year",
    barmode="overlay",
    title="Lap Time Distribution by Season",
    labels={"LapTimeSeconds": "Lap Time (seconds)"},
    opacity=0.7,
    color_discrete_sequence=["#e10600", "#00d2be", "#ff8700"]  # F1 colors
)

fig.update_layout(
    template="plotly_dark",
    bargap=0.1
)

fig.show()

In [ ]:
compound_order = ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]
compounds_present = [c for c in compound_order if c in df_clean["Compound"].unique()]

fig = px.box(
    df_clean[df_clean["Compound"].isin(compounds_present)],
    x="Compound",
    y="LapTimeSeconds",
    color="Compound",
    category_orders={"Compound": compounds_present},
    title="Lap Time Distribution by Tyre Compound",
    labels={"LapTimeSeconds": "Lap Time (seconds)"},
    color_discrete_map={
        "SOFT": "#e10600",
        "MEDIUM": "#ffd700",
        "HARD": "#ffffff",
        "INTERMEDIATE": "#39b54a",
        "WET": "#0067ff"
    }
)

fig.update_layout(template="plotly_dark", showlegend=False)
fig.show()

In [ ]:
# Focus on one track to keep it clean — use Bahrain (Round 1) as example
bahrain = df_clean[
    (df_clean["EventName"].str.contains("Bahrain", case=False)) &
    (df_clean["Year"] == 2023) &
    (df_clean["Compound"].isin(["SOFT", "MEDIUM", "HARD"]))
].copy()

# Group by compound and tyre age, get median lap time
deg_curve = bahrain.groupby(
    ["Compound", "TyreLife"]
)["LapTimeSeconds"].median().reset_index()

fig = px.line(
    deg_curve,
    x="TyreLife",
    y="LapTimeSeconds",
    color="Compound",
    title="Tyre Degradation Curve — Bahrain 2023",
    labels={
        "TyreLife": "Tyre Age (laps)",
        "LapTimeSeconds": "Median Lap Time (seconds)"
    },
    color_discrete_map={
        "SOFT": "#e10600",
        "MEDIUM": "#ffd700",
        "HARD": "#c0c0c0"
    }
)

fig.update_layout(template="plotly_dark")
fig.show()

In [ ]:
# Shows how lap times evolve over the course of a race
# Fuel burn makes cars faster early, tyre deg makes them slower later

race_trend = df_clean[
    (df_clean["EventName"].str.contains("Bahrain", case=False)) &
    (df_clean["Year"] == 2023) &
    (df_clean["Compound"].isin(["SOFT", "MEDIUM", "HARD"]))
].groupby(["LapNumber", "Compound"])["LapTimeSeconds"].median().reset_index()

fig = px.scatter(
    race_trend,
    x="LapNumber",
    y="LapTimeSeconds",
    color="Compound",
    trendline="lowess",
    title="Lap Time Evolution Through Race — Bahrain 2023",
    labels={
        "LapNumber": "Lap Number",
        "LapTimeSeconds": "Median Lap Time (seconds)"
    },
    color_discrete_map={
        "SOFT": "#e10600",
        "MEDIUM": "#ffd700",
        "HARD": "#c0c0c0"
    }
)

fig.update_layout(template="plotly_dark")
fig.show()

In [ ]:
# Select only numeric columns that are relevant
numeric_cols = [
    "LapTimeSeconds", "TyreLife", "LapNumber",
    "AirTemp", "TrackTemp", "Humidity",
    "WindSpeed", "Sector1Seconds", "Sector2Seconds", "Sector3Seconds"
]

# Filter to columns that actually exist in your dataframe
available_cols = [c for c in numeric_cols if c in df_clean.columns]

corr_matrix = df_clean[available_cols].corr()

fig = px.imshow(
    corr_matrix,
    title="Feature Correlation Heatmap",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    text_auto=".2f"
)

fig.update_layout(template="plotly_dark", width=700, height=600)
fig.show()

In [ ]:
laps_per_track = df_clean.groupby(
    ["EventName", "Year"]
)["LapTimeSeconds"].count().reset_index()
laps_per_track.columns = ["Event", "Year", "LapCount"]

fig = px.bar(
    laps_per_track,
    x="Event",
    y="LapCount",
    color="Year",
    barmode="group",
    title="Data Coverage — Laps Per Track Per Season",
    labels={"LapCount": "Number of Laps", "Event": "Race"}
)

fig.update_layout(
    template="plotly_dark",
    xaxis_tickangle=-45,
    height=500
)

fig.show()

In [ ]:
output_path = "../data/processed/laps_cleaned.csv"
df_clean.to_csv(output_path, index=False)

print(f"✅ Cleaned data saved to {output_path}")
print(f"   Shape: {df_clean.shape}")
print(f"   Seasons: {sorted(df_clean['Year'].unique())}")
print(f"   Tracks: {df_clean['EventName'].nunique()}")
print(f"   Total laps: {len(df_clean):,}")